# SPECTRA Rubin RSP Compact-Source Test

This notebook is meant to be run directly on the Rubin Science Platform. It tests one specific compact, cluster-like source in the RSP DP0.2/DC2 object catalog, `dp02_dc2_catalogs.Object`, using its Rubin `objectId`.

The workflow writes a notebook-local config for that source, previews its Rubin photometry, runs the SPECTRA fitter, and inspects the resulting `fit_summary.csv`.

## 1. Set Up the Repository Path

Run this notebook from the repository root or from the `notebooks/` directory.

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
if not (repo_root / "src").exists():
    raise RuntimeError("Run this notebook from the SPECTRA repository root or from notebooks/.")

sys.path.insert(0, str(repo_root))
repo_root

## 2. Imports and RSP Authentication

When this notebook runs on the Rubin Science Platform, it reuses the platform-provided token from the notebook environment. The manual prompt is only a fallback for running the notebook somewhere else.

In [ ]:
import getpass
import yaml
import pandas as pd

from src.cli import validate_config
from src.main import main
from src.data.rubin_query import RubinDataQuery


def configure_rsp_token():
    for env_name in ("RSP_TOKEN", "ACCESS_TOKEN", "JUPYTERHUB_API_TOKEN"):
        token = os.environ.get(env_name)
        if token:
            os.environ["RSP_TOKEN"] = token
            return env_name

    try:
        from rubin_jupyter_utils.lab.notebook import utils as rsp_utils

        token = rsp_utils.get_access_token()
        if token:
            os.environ["RSP_TOKEN"] = token
            return "rubin_jupyter_utils"
    except Exception:
        pass

    os.environ["RSP_TOKEN"] = getpass.getpass("Rubin RSP token: ")
    return "manual prompt"


auth_source = configure_rsp_token()
print(f"RSP authentication configured from: {auth_source}")

## 3. Prepare a Specific Compact-Source Config

This cell creates a small single-object config for a cluster-like DP0.2/DC2 source. The `target_object_id` value can be replaced with another Rubin `objectId` from the same catalog.

In [ ]:
rubin_release_label = "Rubin RSP DP0.2/DC2"
rubin_catalog = "dp02_dc2_catalogs.Object"
target_object_id = 1651281746966115584
rsp_base_url = os.environ.get("EXTERNAL_INSTANCE_URL", "https://data.lsst.cloud").rstrip("/")
rubin_tap_url = f"{rsp_base_url}/api/tap"

config = {
    "input": {
        "type": "rubin_id",
        "rubin_id": target_object_id,
    },
    "rubin": {
        "rsp_token": None,
        "tap_url": rubin_tap_url,
        "catalog": rubin_catalog,
        "flux_type": "psfFlux",
        "bands": ["u", "g", "r", "i", "z", "y"],
    },
    "ssp_model": {
        "type": "fsps",
        "redshift": 0.0,
        "imf": "chabrier",
        "sfh": 0,
        "dust_type": 2,
        "add_neb_emission": False,
    },
    "fitting": {
        "method": "ml",
        "error_floor": 0.05,
        "parameters": ["mass", "age", "metallicity"],
        "priors": {
            "mass": [6.0, 12.0],
            "age": [0.01, 13.5],
            "metallicity": [-2.0, 0.5],
        },
    },
    "plotting": {
        "output_dir": str(repo_root / "outputs" / f"notebook_dp02_compact_{target_object_id}"),
        "show_plots": False,
        "save_plots": True,
        "formats": ["png"],
        "dpi": 150,
    },
    "output": {
        "save_photometry": True,
    },
}

notebook_config_path = repo_root / "rsp_config.yaml"
notebook_config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

print(f"Testing {rubin_release_label} compact-source objectId: {target_object_id}")
print(f"Catalog: {rubin_catalog}")
print(f"TAP URL: {rubin_tap_url}")
print(notebook_config_path)
config

## 4. Validate the Config

This catches missing required sections before sending a query to Rubin.

In [ ]:
status = validate_config(str(notebook_config_path))
if status != 0:
    raise RuntimeError(f"Config validation failed with status {status}")

print("Config validation passed.")

## 5. Preview the Target Source Photometry

This quick preview checks that RSP authentication can retrieve the target `objectId`, then extracts the Rubin bands SPECTRA will fit.

In [ ]:
query = RubinDataQuery(config=config)
object_table = query.query_object(
    config["input"]["rubin_id"],
    catalog=config["rubin"]["catalog"],
)
phot_data = query.extract_photometry(
    object_table,
    flux_type=config["rubin"].get("flux_type", "psfFlux"),
    bands=config["rubin"].get("bands"),
)

print(f"Retrieved objectId {config['input']['rubin_id']} from {config['rubin']['catalog']}.")
print(f"Bands available for fitting: {phot_data.get('bands')}")
print(f"Flux array shape: {phot_data['obs_flux'].shape}")
object_table[[col for col in ["objectId", "coord_ra", "coord_dec"] if col in object_table.columns]].head()

## 6. Run SPECTRA

This runs maximum-likelihood fitting for the target compact source and writes plots plus a summary table under `outputs/notebook_dp02_compact_<objectId>/`.

In [ ]:
main(str(notebook_config_path))

## 7. Inspect Fit Summary

The table below is the first pass quality check. Pay closest attention to `chi2_red`, parameter values at prior boundaries, and whether any object produced missing values.

In [ ]:
summary_path = Path(config["plotting"]["output_dir"]) / "fit_summary.csv"
summary = pd.read_csv(summary_path)
summary

## 8. Quick Diagnostics

This cell flags high reduced chi-square values and lists the per-object output folders.

In [ ]:
display_cols = [col for col in ["object_id", "redshift", "chi2_red", "mass", "age", "metallicity", "dust"] if col in summary.columns]
display(summary[display_cols])

poor = summary.loc[summary["chi2_red"] > 2, display_cols]
if len(poor):
    print("Objects with chi2_red > 2; inspect residual plots before using these scientifically:")
    display(poor)
else:
    print("All objects have chi2_red <= 2 in this smoke test.")

for path in sorted(Path(config["plotting"]["output_dir"]).glob("*")):
    if path.is_dir():
        print(path.relative_to(repo_root))